# Identifying Quasi-Periodic Oscillation (QPO) Candidates

### STATiX Variable Source Catalogue — Pouliasis et al. (A&A)

This notebook demonstrates how to identify candidate sources exhibiting recurrent, quasi-periodic flux modulations in the **published** STATiX variable source catalogue (`STATIX_VC_published.fits`, referred to as `out` below).


**Important caveat:** the flags built here (multi-peak, block count, repeated detections) are *necessary but not sufficient* conditions for a QPO. They do **not** by themselves constitute a significance test. A rigorous QPO claim requires extracting full-resolution light curves from the archive and applying the periodogram analysis of Sect. 5.3, including trial-factor correction — this notebook only narrows down the parent sample to promising candidates for that follow-up.

This notebook uses **only the columns available in the public published table** — no access to internal pipeline products or raw archival cubes is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.table import Table, unique
from astropy.time import Time
from astropy.stats import mad_std as mad

## 1. Load the published catalogue

In [ ]:
out = Table.read("STATIX_VC_published.fits")
print(f"Loaded {len(out)} detections with {len(out.colnames)} columns.")
out.colnames

## 2. Select sources with multiple flux peaks

We start from `FLAG_MULTIPEAK`, which is `True` if any detector (pn, M1, or M2) shows more than one significant peak in its Bayesian-block light curve (see Appendix D of the paper). This is the basic building block of oscillatory-looking behaviour.

In [ ]:
q = out[out["FLAG_MULTIPEAK"]]
print(f"Multi-peak detections: {len(q)}")

### Number of Bayesian blocks

The published table does not include a precomputed block-count column, but it can be recovered directly from the length of the `pn_LC_BB` array for each detection (one row per Bayesian block; see the catalogue column description). We require at least 10 pn blocks, i.e. a light curve resolved into enough independent flux segments to plausibly trace a recurring pattern rather than a single isolated flare or dip.

In [ ]:
def n_blocks(bb_col_row):
    """Number of Bayesian blocks stored in a pn_LC_BB / M1_LC_BB / M2_LC_BB row."""
    if bb_col_row is None:
        return 0
    arr = np.ma.filled(np.atleast_2d(bb_col_row), np.nan)
    # drop fully-NaN padding rows (ragged-array storage)
    valid = ~np.isnan(arr).all(axis=1)
    return int(valid.sum())

n_blocks_pn = np.array([n_blocks(row) for row in q["pn_LC_BB"]])

q5 = q[n_blocks_pn >= 10]
print(f"Multi-peak detections with >=10 pn Bayesian blocks: {len(q5)}")

## 3. Split by source classification

`CLASS_RF` follows the same convention as `CLASS` (Random-Forest classification):

| Value | Meaning |
|---|---|
| -1 | unclassified |
| 0 | secure Galactic |
| 1 | likely Galactic |
| 2 | likely extragalactic |
| 3 | secure extragalactic |

QPOs discussed in this paper concern the **extragalactic** population (AGN and TDEs), so we isolate that subset.

In [ ]:
extra = q5[(q5["CLASS"] == 2) | (q5["CLASS"] == 3)]
stars = q5[(q5["CLASS"] == 0) | (q5["CLASS"] == 1)]
unclas = q5[q5["CLASS"] == -1]

print(f"Extragalactic candidates : {len(extra)}")
print(f"Galactic candidates      : {len(stars)}")
print(f"Unclassified candidates  : {len(unclas)}")

### Unique sources

The catalogue is detection-based: the same physical source observed in multiple XMM-Newton pointings appears as multiple rows sharing the same `group_id`. We reduce each class to unique sources for a fair source count.

In [ ]:
starsU = unique(stars, keys="group_id")
extraU = unique(extra, keys="group_id")
unclasU = unique(unclas, keys="group_id")

print(f"Unique Galactic sources     : {len(starsU)}")
print(f"Unique extragalactic sources: {len(extraU)}")
print(f"Unique unclassified sources : {len(unclasU)}")

print(f"Total candidates : {len(unclasU)+len(extraU)+len(starsU)}")

## 4. Focus on extragalactic sources detected in one or more observations

*** `group_number > 1`, i.e. detected in at least one additional observation besides the one contributing this row.

In [ ]:
gg = extraU[extraU["group_number"] >= 1]
print(f"Extragalactic multi-epoch QPO candidates: {len(gg)}")
gg["DETID", "group_id", "group_number", "CLASS_RF", "RA_X", "DEC_X"]

## 5. Visualise candidate light curves

The function below plots the Bayesian-block light curve of every observation belonging to a given `group_id`, using **only the block start/end times stored in `pn_LC_BB`** (columns: start time, end time, number of frames, source counts, background counts, significance flag; see catalogue description). The raw per-frame light curve (`pn_LC`) does not carry an absolute timestamp in the published table, so the block-level representation is used to place each observation on a common, continuous time axis.

Times are stored in XMM Mission Elapsed Time (MET) seconds, referenced to 1998-01-01 00:00:00 (TT).

In [ ]:
MET_REF = Time("1998-01-01 00:00:00", scale="tt")

def _clean_bb(bb_row):
    """Return a clean (N, 6) array of valid pn Bayesian blocks from a pn_LC_BB row."""
    arr = np.ma.filled(np.atleast_2d(bb_row), np.nan)
    valid = ~np.isnan(arr).any(axis=1)
    arr = arr[valid]
    return arr if arr.shape[0] > 0 else None


def _reconstruct_frame_times(lc_bb):
    """
    Approximate per-frame times from Bayesian-block start/end times,
    assuming frames are equally spaced within each block. Needed because
    the published pn_LC array stores only counts, with no timestamps.
    """
    frame_times = []
    for jj in range(lc_bb.shape[0]):
        start, end, n_frames_block = lc_bb[jj, 0], lc_bb[jj, 1], lc_bb[jj, 2]
        n_frames_block = int(n_frames_block) if n_frames_block > 0 else 1
        dt = (end - start) / n_frames_block
        block_times = start + dt * (np.arange(n_frames_block) + 0.5)
        frame_times.append(block_times)
    return np.concatenate(frame_times)

def _prepare_lc(row, inst):
    """Same logic as _prepare_fxt_lc, generalized to any instrument ('pn','M1','M2')."""
    bb_col = f"{inst}_LC_BB"
    lc_col = f"{inst}_LC"

    bb = _clean_bb(row[bb_col])
    if bb is None:
        return None
    bb = bb[np.argsort(bb[:, 0])]

    raw_lc = np.ma.filled(np.atleast_2d(row[lc_col]), np.nan)
    mask_lc = ~np.isnan(raw_lc[:, 0])
    lc = raw_lc[mask_lc]
    if lc.size == 0:
        return None

    frame_times = _reconstruct_frame_times(bb)
    n_use = min(len(frame_times), len(lc))
    frame_times = frame_times[:n_use]
    lc = lc[:n_use]

    t0 = bb[0, 0]
    time_raw = frame_times - t0

    rate_bb = bb[:, 3] / np.where(bb[:, 2] == 0, 1, bb[:, 2])
    return time_raw, lc, bb, rate_bb, t0


def _prepare_group_lc(row, instrument_priority=("pn", "M1", "M2")):
    """Try each instrument in priority order; return the first usable (time_raw, lc, bb, rate_bb, t0, inst)."""
    for inst in instrument_priority:
        prepared = _prepare_lc(row, inst)
        if prepared is not None:
            time_raw, lc, bb, rate_bb, t0 = prepared
            return time_raw, lc, bb, rate_bb, t0, inst
    return None


def plot_group_lc(out, group_id, n_rows=4, n_cols=5, instrument_priority=("pn", "M1", "M2")):
    """
    Plot the raw + Bayesian-block light curve of every observation belonging to
    `group_id`, placed on a common, continuous time axis.

    Uses only columns available in the published `out` table:
    OBSID, and pn_LC/pn_LC_BB (falls back to M1/M2, in `instrument_priority` order).
    """
    group_rows = out[out["group_id"] == group_id]
    if len(group_rows) == 0:
        print(f"No rows found for group_id={group_id}")
        return

    
    all_data = []
    for row in group_rows:
        obsid = row["OBSID"]

        prepared = _prepare_group_lc(row, instrument_priority)
        if prepared is None:
            continue
        time_raw, lc, bb, rate_bb, t0, inst = prepared

        obs_date_str = (MET_REF + t0 / 86400.0).datetime.strftime("%Y-%m-%d")

        all_data.append({
            "obsid": obsid,
            "obs_date_str": obs_date_str,
            "time_raw": time_raw,
            "lc": lc,
            "bb": bb,
            "rate_bb": rate_bb,
            "t0": t0,
            "inst": inst,
            "abs_time_val": t0,
        })

    if not all_data:
        print(f"No valid light-curve data found for group_id={group_id}")
        return

    all_data = sorted(all_data, key=lambda x: x["abs_time_val"])
    t_ref = all_data[0]["abs_time_val"]  # common time-axis origin

    # common y-scale from both raw counts and BB rates across all panels
    ymax = max(
        max(d["lc"][:, 0].max(), d["rate_bb"].max())
        for d in all_data
    ) * 1.1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))
    axes = np.atleast_1d(axes).flatten()

    for idx, info in enumerate(all_data):
        if idx >= n_rows * n_cols:
            break
        ax = axes[idx]

        # shift this obs's local time_raw (relative to its own t0) onto the shared axis
        offset_days = (info["t0"] - t_ref) / 86400.0
        time_rel = info["time_raw"] / 86400.0 + offset_days

        lc = info["lc"]
        bb = info["bb"]
        rate_bb = info["rate_bb"]

        # raw light curve: background (gray +) and source (red steps/stars)
        ax.scatter(time_rel, lc[:, 1], color="gray", alpha=0.6, zorder=1, marker="+", s=10)
        ax.plot(time_rel, lc[:, 0], color="red", alpha=0.5, drawstyle="steps-mid", lw=1, zorder=1)
        ax.scatter(time_rel, lc[:, 0], color="red", alpha=0.8, zorder=1, marker="*", s=10)

        # Bayesian blocks
        for jj in range(bb.shape[0]):
            start = (bb[jj, 0] - t_ref) / 86400.0
            end = (bb[jj, 1] - t_ref) / 86400.0
            rate = rate_bb[jj]
            is_sig = bb[jj, 5] == 1
            ax.plot([start, end], [rate, rate], color="black" if is_sig else "orange",
                     lw=1.5, solid_capstyle="butt", zorder=3)
            if is_sig:
                ax.axvspan(start, end, color="gray", alpha=0.1, zorder=2)

        ax.set_title(f"{info['obs_date_str']}\n{info['obsid']} ({info['inst']})", fontsize=7)
        ax.set_ylim(0, ymax)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
        ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
        ax.tick_params(axis="both", labelsize=6)
        ax.grid(True, alpha=0.1)

    for i in range(len(all_data), n_rows * n_cols):
        axes[i].axis("off")

    fig.suptitle(f"group_id = {group_id}", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"LC_{group_id}.png", dpi=150)
    plt.show()

In [ ]:
for i in range(len(gg)):
    gid = gg["group_id"][i]
    print(f"group_id = {gid}  (DETID = {gg['DETID'][i]})")
    plot_group_lc(out, group_id=gid, n_rows=4, n_cols=5)

## Next steps

The sources plotted above are **candidates** requiring dedicated follow-up: a Lomb-Scargle periodogram analysis against a red-noise + Poisson continuum, with significance assessed via trial-factor-corrected thresholds, as described in Sect. 5.3 of Pouliasis et al. This notebook only performs the coarse, catalogue-level pre-selection; the full periodogram analysis requires higher time-resolution light curves extracted directly from the XMM-Newton archive and is beyond the scope of the released columns in the published catalogue.